# Bronze Ingest: NYC Taxi Trip Data

Downloads a public NYC Yellow Taxi trip-record parquet file and persists it as a managed Spark table, unmodified from source. This is the `bronze` layer — later notebooks in `data-transformation/` build cleaned/aggregated tables on top of it.

Re-runnable: uses `CREATE DATABASE IF NOT EXISTS` and `mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Configuration

Change these to point at a different month/dataset without touching the logic below.

In [ ]:
DATABASE_NAME = "nyc_taxi"
TABLE_NAME = "yellow_tripdata"
SOURCE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
LOCAL_FILE = "/tmp/yellow_tripdata_2026_01.parquet"

## Download source file

In [ ]:
import urllib.request

urllib.request.urlretrieve(SOURCE_URL, LOCAL_FILE)

## Load dataset

In [ ]:
df = spark.read.parquet(LOCAL_FILE)

## Create schema/database

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'NYC Taxi Trip Record datasets used for analytics demonstrations'
""")

## Persist as a managed Spark table

In [ ]:
df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TABLE_NAME}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_trips FROM {DATABASE_NAME}.{TABLE_NAME}").show()
spark.sql(f"DESCRIBE {DATABASE_NAME}.{TABLE_NAME}").show(truncate=False)